In [2]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

print("📥 1. Loading and formatting data...")
hf_dataset = load_dataset("lanretto/discourse_quality", data_files="full_labeled_flattened.csv", split="train")
df = hf_dataset.to_pandas()

target_cols = [
    'level_of_justification', 'respect_towards_demands', 'respect_towards_counterarguments',
    'content_of_justification', 'respect_towards_groups', 'constructive_politics'
]
df_model = df[['comment'] + target_cols].dropna()
df_model['comment'] = df_model['comment'].astype(str)

X = df_model['comment']
y = df_model[target_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🔠 2. Fitting TF-IDF Vectorizer (max 10,000 words)...")
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("🌲 3. Training Multi-Output Random Forest (This might take a while!)...")
# n_jobs=-1 tells it to use all available CPU cores
# We force the trees to stop growing at a depth of 15!
rf_base = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
#rf_base = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
model_rf = MultiOutputRegressor(rf_base)
model_rf.fit(X_train_tfidf, y_train)

print("📊 4. Calculating Training and Validation Loss (MSE)...")
y_train_pred = model_rf.predict(X_train_tfidf)
y_test_pred = model_rf.predict(X_test_tfidf)

print("\n--- RANDOM FOREST RESULTS ---")
total_train_mse = 0
total_val_mse = 0

for i, col in enumerate(target_cols):
    train_mse = mean_squared_error(y_train.iloc[:, i], y_train_pred[:, i])
    val_mse = mean_squared_error(y_test.iloc[:, i], y_test_pred[:, i])
    
    total_train_mse += train_mse
    total_val_mse += val_mse
    
    print(f"{col}:")
    print(f"   Train MSE: {train_mse:.4f}  |  Val MSE: {val_mse:.4f}")

print(f"\nOVERALL TRAIN MSE: {total_train_mse / 6:.4f}")
print(f"OVERALL VAL MSE:   {total_val_mse / 6:.4f}")

📥 1. Loading and formatting data...
🔠 2. Fitting TF-IDF Vectorizer (max 10,000 words)...
🌲 3. Training Multi-Output Random Forest (This might take a while!)...
📊 4. Calculating Training and Validation Loss (MSE)...

--- RANDOM FOREST RESULTS ---
level_of_justification:
   Train MSE: 0.1054  |  Val MSE: 0.1134
respect_towards_demands:
   Train MSE: 0.1074  |  Val MSE: 0.1134
respect_towards_counterarguments:
   Train MSE: 0.0972  |  Val MSE: 0.1036
content_of_justification:
   Train MSE: 0.0932  |  Val MSE: 0.0975
respect_towards_groups:
   Train MSE: 0.0566  |  Val MSE: 0.0585
constructive_politics:
   Train MSE: 0.0829  |  Val MSE: 0.0858

OVERALL TRAIN MSE: 0.0905
OVERALL VAL MSE:   0.0954
